In [48]:
import os
os.chdir('/home/hrayrh/projects/incontext_spurious')
os.environ['DATA_ROOT_DIR'] = '/home/hrayrh/data'

In [49]:
import sys
sys.path.append('/home/hrayrh/projects/incontext_spurious')

In [50]:
from collections import Counter
from matplotlib import pyplot as plt
%matplotlib inline
import numpy as np
from sklearn.linear_model import LogisticRegression
from src.datamodules.datasets import MultiNLIForEncodingExtraction, MultiNLIExtracted

# Inspect MultiNLIForEncodingExtraction

In [51]:
ds = MultiNLIForEncodingExtraction(root_dir='/home/hrayrh/data/')
train = ds.get_subset(split='train')

In [19]:
len(train), len(train) == len(train.sentence_pairs)

(321373, True)

In [20]:
train[0]

(('One of our number will carry out your instructions minutely.',
  'A member of my team will execute your orders with immense precision.'),
 2)

In [25]:
val = ds.get_subset(split='val')
len(val)

71289

# Inspect extracted files

In [32]:
saved_data = np.load('/home/hrayrh/data/multinli/bert/test/combined.npz')
saved_data.keys()

KeysView(NpzFile '/home/hrayrh/data/multinli/bert/test/combined.npz' with keys: encodings, indices_map)

In [33]:
saved_data['encodings'].shape

(9815, 768)

In [34]:
len(saved_data['indices_map'])

402477

In [35]:
saved_data['indices_map'][:100]

array([-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1])

In [36]:
saved_data['indices_map'][-100:]

array([9715, 9716, 9717, 9718, 9719, 9720, 9721, 9722, 9723, 9724, 9725,
       9726, 9727, 9728, 9729, 9730, 9731, 9732, 9733, 9734, 9735, 9736,
       9737, 9738, 9739, 9740, 9741, 9742, 9743, 9744, 9745, 9746, 9747,
       9748, 9749, 9750, 9751, 9752, 9753, 9754, 9755, 9756, 9757, 9758,
       9759, 9760, 9761, 9762, 9763, 9764, 9765, 9766, 9767, 9768, 9769,
       9770, 9771, 9772, 9773, 9774, 9775, 9776, 9777, 9778, 9779, 9780,
       9781, 9782, 9783, 9784, 9785, 9786, 9787, 9788, 9789, 9790, 9791,
       9792, 9793, 9794, 9795, 9796, 9797, 9798, 9799, 9800, 9801, 9802,
       9803, 9804, 9805, 9806, 9807, 9808, 9809, 9810, 9811, 9812, 9813,
       9814])

# Inspecting MultiNLIExtracted to see how well Y and S are predictable from encodings

**Stroberta**


512 examples
* Y acc = 96.4
* S acc = 92.9

full dataset (320k examples)
* Y acc = 96.7
* S acc = 94.8

**Bert**

512 examples
* Y acc = 63.9
* S acc = 92.6

full dataset (320k examples)
* Y acc@full = 73.1
* S acc@full = 95.3

**RoBERTa (base)**

512 examples
* Y acc = 66.8
* S acc = 92.8

full dataset (320k examples)
* Y acc = 73.5
* S acc = 94.4

**RoBERTa (large)**

512 examples
* Y acc = 66.1
* S acc = 92.9

full dataset (320k examples)
* Y acc = 75.7
* S acc = 95.0

In [86]:
ds = MultiNLIExtracted(
    dataset_path='/home/hrayrh/data',
    # encoding_extractor='bert',
    # encoding_extractor='stroberta',
    encoding_extractor='roberta',
    # encoding_extractor='roberta_large',
)

In [87]:
train = ds.get_subset('train')
val = ds.get_subset('val')
test = ds.get_subset('test')
print(f'{len(train)=}, {len(val)=}, {len(test)=}')

len(train)=321373, len(val)=71289, len(test)=9815


In [88]:
train_X, examples = train[np.arange(len(train))]
print(f'{train_X.shape=}, {examples.shape=}')
train_s = examples[:, 1]
train_y = examples[:, 2]
print(f'{train_s.shape=}, {train_y.shape=}')

train_X.shape=(321373, 768), examples.shape=(321373, 3)
train_s.shape=(321373,), train_y.shape=(321373,)


In [89]:
val_X, examples = val[np.arange(len(val))]
print(f'{val_X.shape=}, {examples.shape=}')
val_s = examples[:, 1]
val_y = examples[:, 2]
print(f'{val_s.shape=}, {val_y.shape=}')

val_X.shape=(71289, 768), examples.shape=(71289, 3)
val_s.shape=(71289,), val_y.shape=(71289,)


In [90]:
test_X, examples = test[np.arange(len(test))]
print(f'{test_X.shape=}, {examples.shape=}')
test_s = examples[:, 1]
test_y = examples[:, 2]
print(f'{test_s.shape=}, {test_y.shape=}')

test_X.shape=(9815, 768), examples.shape=(9815, 3)
test_s.shape=(9815,), test_y.shape=(9815,)


## Predicting Y

In [103]:
log_reg = LogisticRegression(max_iter=10000)

selected_indices = np.random.choice(np.arange(len(train_X)), 512, replace=False)
log_reg.fit(train_X[selected_indices], train_y[selected_indices])

# log_reg.fit(train_X, train_y)

val_pred = log_reg.predict(val_X)
val_acc = np.mean(val_pred == val_y)
print(f'predicting Y, {val_acc=}')

predicting Y, val_acc=0.6685323121379175


## Predicting S

In [85]:
log_reg = LogisticRegression(max_iter=10000)

# selected_indices = np.random.choice(np.arange(len(train_X)), 512, replace=False)
# log_reg.fit(train_X[selected_indices], train_s[selected_indices])

log_reg.fit(train_X, train_s)

val_pred = log_reg.predict(val_X)
val_acc = np.mean(val_pred == val_s)
print(f'predicting S, {val_acc=}') 

predicting S, val_acc=0.9500063123343012


## Imbalanced contexts

In [13]:
def worst_group_acc(pred, y, s):
    g = 2 * y + s
    accs = []
    for g_idx in range(4):
        mask = (g == g_idx)
        g_acc = np.mean(pred[mask] == y[mask])
        accs.append(g_acc)
    print('group accuracies:', accs)
    print('overall accuracy:', np.mean(pred==y))
    return np.min(accs)

In [14]:
train_X, examples = train[np.arange(len(train))]
print(f'{train_X.shape=}, {examples.shape=}')
train_s = examples[:, 1]
train_y = examples[:, 2]
maj_mask = (train_s == train_y)
min_mask = (train_s != train_y)
print('maj min exmaples before', np.sum(maj_mask), np.sum(min_mask))
min_ratio = 1.0 / 9  # produces [0.45, 0.05, 0.05, 0.45]
# min_ratio = 0.0  # produces [0.5, 0.0, 0.0, 0.5]
select_min_indices = np.random.choice(
    np.where(min_mask)[0], 
    size=int(np.sum(min_mask) * (1-min_ratio)),
    replace=False)
min_mask[select_min_indices] = False
print('maj min exmaples after', np.sum(maj_mask), np.sum(min_mask))
mask = (maj_mask | min_mask)

print(f'{train_s.shape=}, {train_y.shape=}')

# normal ERM
log_reg = LogisticRegression(max_iter=1000)    
log_reg.fit(train_X[mask], train_y[mask])

# downsampled ERM
if False:
    log_reg = LogisticRegression(max_iter=1000)    
    X = train_X[mask]
    y = train_y[mask]
    s = train_s[mask]
    g = 2 * y + s
    counts = Counter(g)
    print(counts)
    min_count = np.min([counts[g_idx] for g_idx in counts])
    selected_indices = []
    for g_idx in range(4):
        selected_indices.append(
            np.random.choice(np.where(g==g_idx)[0], size=min_count, replace=False)
        )
    selected_indices=np.concatenate(selected_indices, axis=0)
    log_reg.fit(X[selected_indices], y[selected_indices])

train_X.shape=(321373, 768), examples.shape=(321373, 3)
maj min exmaples before 226276 95097
maj min exmaples after 226276 10567
train_s.shape=(321373,), train_y.shape=(321373,)


In [15]:
val_pred = log_reg.predict(val_X)
print(f'predicting Y, matching Y, wg acc: {worst_group_acc(val_pred, val_y, val_s)}')

group accuracies: [0.9876620300589983, 0.6465661641541038, 0.09192270823873669, 0.6289144050104384]
overall accuracy: 0.7136164064582193
predicting Y, matching Y, wg acc: 0.09192270823873669


In [16]:
val_pred = log_reg.predict(val_X)
print(f'predicting Y, matching S, wg acc: {worst_group_acc(val_pred, val_s, val_y)}')

group accuracies: [0.9876620300589983, 0.9080772917612633, 0.35343383584589616, 0.6289144050104384]
overall accuracy: 0.9356282175370674
predicting Y, matching S, wg acc: 0.35343383584589616
